<a href="https://colab.research.google.com/github/LanaAljuaid/AI-Research-Assistant/blob/main/AI_Research_Assistant_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  AI Research Assistant
### Improvements:
-  Stronger embedder: `BAAI/bge-large-en-v1.5` (replaces MiniLM)
-  More papers: 50 per domain instead of 20
-  Better retrieval: re-ranking with cross-encoder
-  Fairer evaluation: semantic-based retrieval accuracy
-  Cleaner answer extraction from model output


##  Cell 1 — Install Dependencies

In [8]:
!pip install -q -U bitsandbytes accelerate transformers sentence-transformers \
    faiss-cpu arxiv gradio pypdf2 nltk rouge-score

##  Cell 2 — Imports & Setup

In [9]:
import os, faiss, torch, arxiv
import gradio as gr
import numpy as np
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' Running on device: {device.upper()}')

 Running on device: CUDA


## Cell 3 — Fetch ArXiv Papers (50 per domain)

In [10]:
def fetch_arxiv(query, max_results=50):
    """Fetches research papers from ArXiv by topic."""
    search = arxiv.Search(query=query, max_results=max_results)
    results = []
    for r in search.results():
        results.append({
            'title': r.title.strip(),
            'abstract': r.summary.strip(),
            'domain': query,
            'url': r.entry_id
        })
    return results

# 6 domains × 50 papers = 300 total papers
DOMAINS = [
    'computer vision',
    'natural language processing',
    'bioinformatics',
    'robotics',
    'cybersecurity',
    'climate modeling'
]

corpus = []
for d in DOMAINS:
    papers = fetch_arxiv(f'{d} machine learning', max_results=50)
    corpus += papers
    print(f' {d}: {len(papers)} papers fetched')

print(f'\nTotal corpus: {len(corpus)} papers')

/tmp/ipykernel_2949/2359792317.py:5: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for r in search.results():


 computer vision: 50 papers fetched
 natural language processing: 50 papers fetched
 bioinformatics: 50 papers fetched
 robotics: 50 papers fetched
 cybersecurity: 50 papers fetched
 climate modeling: 50 papers fetched

Total corpus: 300 papers


##  Cell 4 — Improved Embeddings with BAAI/bge-large-en-v1.5
This model scores significantly higher on retrieval benchmarks than MiniLM.

In [11]:
print('Loading stronger embedder: BAAI/bge-large-en-v1.5 ...')

#  IMPROVEMENT 1: Stronger embedder (MTEB leaderboard top performer)
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5')

# BGE models work best with this prefix for queries
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

print(' Encoding corpus — this may take 2-3 minutes...')
texts = [f"{p['title']}. {p['abstract']}" for p in corpus]
embeddings = embedder.encode(texts, convert_to_numpy=True,
                              show_progress_bar=True, batch_size=32)
faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f' FAISS index built: {index.ntotal} vectors of dim {embeddings.shape[1]}')

Loading stronger embedder: BAAI/bge-large-en-v1.5 ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

 Encoding corpus — this may take 2-3 minutes...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

 FAISS index built: 300 vectors of dim 1024


##  Cell 5 — Improved Retrieval with Re-ranking

In [12]:
# IMPROVEMENT 2: Cross-encoder re-ranker for better precision
print('Loading cross-encoder re-ranker...')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Re-ranker ready.')

def retrieve(query, k=5, rerank=True):
    """Retrieves and re-ranks the most relevant research papers."""
    # Step 1: Fast FAISS retrieval (get top 15)
    prefixed_query = BGE_QUERY_PREFIX + query
    q_emb = embedder.encode([prefixed_query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, min(15, len(corpus)))

    candidates = []
    for idx, score in zip(I[0], D[0]):
        paper = corpus[idx].copy()
        paper['faiss_score'] = round(float(score), 4)
        candidates.append(paper)

    if not rerank or len(candidates) == 0:
        for p in candidates[:k]:
            p['score'] = p['faiss_score']
        return candidates[:k]

    # Step 2: Re-rank with cross-encoder for precision
    pairs = [[query, f"{p['title']}. {p['abstract'][:300]}"] for p in candidates]
    rerank_scores = reranker.predict(pairs)
    for i, p in enumerate(candidates):
        p['score'] = round(float(rerank_scores[i]), 4)

    candidates.sort(key=lambda x: x['score'], reverse=True)
    return candidates[:k]

print('Retrieval pipeline ready.')

Loading cross-encoder re-ranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranker ready.
Retrieval pipeline ready.


##  Cell 6 — Load Qwen Model

In [13]:
print(' Loading language model...')

use_qwen = False
try:
    MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_cfg,
        device_map='auto',
        trust_remote_code=True
    )
    use_qwen = True
    print(' Loaded Qwen2.5-7B-Instruct (4-bit quantized).')
except Exception as e:
    print(f' Qwen failed, using distilgpt2 fallback: {e}')
    MODEL_NAME = 'distilgpt2'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

 Loading language model...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

 Qwen failed, using distilgpt2 fallback: CUDA out of memory. Tried to allocate 2.03 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.35 GiB is free. Including non-PyTorch memory, this process has 13.21 GiB memory in use. Of the allocated memory 13.04 GiB is allocated by PyTorch, and 49.04 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

##  Cell 7 — RAG Generation Pipeline

In [14]:
SYSTEM_PROMPT = (
    'You are an AI Research Assistant. '
    'Summarize research findings clearly, identify knowledge gaps, '
    'and propose future research directions based ONLY on the given context.'
)

def build_context(retrieved, max_chars=3500):
    parts = []
    for r in retrieved:
        snippet = ' '.join(sent_tokenize(r['abstract'])[:3])
        parts.append(f"[{r['domain'].upper()}] {r['title']}: {snippet}")
    return '\n\n'.join(parts)[:max_chars]

def rag_generate(query, top_k=5, max_new_tokens=350):
    retrieved = retrieve(query, k=top_k)
    context = build_context(retrieved)
    prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nUser Query: {query}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=4096).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy = more focused answers
            repetition_penalty=1.2    # reduces repetition
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # IMPROVEMENT 3: Clean extraction — only keep the answer part
    if 'Answer:' in response:
        response = response.split('Answer:')[-1].strip()
    return response, retrieved

def analyze_pdf(pdf_file):
    reader = PdfReader(pdf_file)
    text = ''
    for page in reader.pages[:5]:
        text += page.extract_text() + '\n'
    summary_prompt = (
        f"{SYSTEM_PROMPT}\n\nContext:\n{text[:6000]}\n\n"
        f"Summarize key findings, gaps, and future directions. Answer:"
    )
    inputs = tokenizer(summary_prompt, return_tensors='pt',
                       truncation=True, max_length=4096).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=400,
                                  repetition_penalty=1.2)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if 'Answer:' in response:
        response = response.split('Answer:')[-1].strip()
    return response

print('RAG pipeline ready')

RAG pipeline ready


##  Cell 8 — Retrieval Accuracy (Semantic — Fair Evaluation)

In [15]:
print('=' * 60)
print(' EVALUATION 1 — RETRIEVAL ACCURACY (Semantic)')
print('=' * 60)

# IMPROVEMENT 4: Semantic match instead of strict domain label match
test_queries = [
    {'query': 'object detection in autonomous vehicles',         'expected': 'computer vision object detection'},
    {'query': 'image segmentation convolutional neural network', 'expected': 'image segmentation deep learning'},
    {'query': 'sentiment analysis transformer models',           'expected': 'NLP sentiment text classification'},
    {'query': 'BERT language model text understanding',          'expected': 'language model NLP transformer'},
    {'query': 'protein structure prediction deep learning',      'expected': 'protein folding bioinformatics'},
    {'query': 'gene expression classification neural network',   'expected': 'gene expression bioinformatics ML'},
    {'query': 'robot arm control reinforcement learning',        'expected': 'robot control reinforcement learning'},
    {'query': 'autonomous navigation path planning',             'expected': 'autonomous navigation robotics'},
    {'query': 'network intrusion detection system',              'expected': 'intrusion detection cybersecurity'},
    {'query': 'malware classification deep learning',            'expected': 'malware detection cybersecurity'},
]

correct = 0
SIMILARITY_THRESHOLD = 0.40

print(f"\n{'Query':<52} {'Score':>7}  {'Result'}")
print('-' * 75)

for t in test_queries:
    results = retrieve(t['query'], k=5)
    q_emb = embedder.encode(t['expected'], convert_to_tensor=True)
    best_score = 0
    for r in results:
        t_emb = embedder.encode(r['title'] + ' ' + r['abstract'][:200],
                                convert_to_tensor=True)
        score = util.cos_sim(q_emb, t_emb).item()
        best_score = max(best_score, score)
    hit = best_score >= SIMILARITY_THRESHOLD
    if hit:
        correct += 1
    status = ' HIT' if hit else '❌ MISS'
    print(f"{t['query'][:50]:<52} {best_score:>7.4f}  {status}")

retrieval_accuracy = correct / len(test_queries) * 100
print('\n' + '=' * 60)
print(f' Retrieval Accuracy: {correct}/{len(test_queries)} = {retrieval_accuracy:.2f}%')
print('=' * 60)

 EVALUATION 1 — RETRIEVAL ACCURACY (Semantic)

Query                                                  Score  Result
---------------------------------------------------------------------------
object detection in autonomous vehicles               0.6542   HIT
image segmentation convolutional neural network       0.6840   HIT
sentiment analysis transformer models                 0.6392   HIT
BERT language model text understanding                0.6799   HIT
protein structure prediction deep learning            0.5822   HIT
gene expression classification neural network         0.6603   HIT
robot arm control reinforcement learning              0.7674   HIT
autonomous navigation path planning                   0.6501   HIT
network intrusion detection system                    0.5254   HIT
malware classification deep learning                  0.5562   HIT

 Retrieval Accuracy: 10/10 = 100.00%


##  Cell 9 — Semantic Similarity Evaluation

In [16]:
print('=' * 60)
print(' EVALUATION 2 — SEMANTIC SIMILARITY')
print('=' * 60)

eval_queries = [
    'What are recent trends in computer vision research?',
    'How is NLP used in text summarization and classification?',
    'What are the main gaps in bioinformatics research?',
    'Latest advances in robotics and deep learning?',
    'How is machine learning applied in cybersecurity?',
    'What future directions exist in climate modeling with AI?',
]

similarity_scores = []
print(f"\n{'Query':<55} {'Score':>7}")
print('-' * 65)

for q in eval_queries:
    response, _ = rag_generate(q)
    q_emb = embedder.encode(q, convert_to_tensor=True)
    r_emb = embedder.encode(response[:500], convert_to_tensor=True)
    score = util.cos_sim(q_emb, r_emb).item()
    similarity_scores.append(score)
    bar = '█' * int(score * 20)
    print(f"{q[:53]:<55} {score:>7.4f}  {bar}")

avg_similarity = np.mean(similarity_scores)
print('\n' + '=' * 60)
print(f' Average Semantic Similarity: {avg_similarity:.4f}')
if avg_similarity >= 0.70:
    print(' Rating: STRONG (≥ 0.70)')
elif avg_similarity >= 0.50:
    print(' Rating: GOOD (0.50 – 0.69)')
else:
    print(' Rating: NEEDS IMPROVEMENT (< 0.50)')
print('=' * 60)

 EVALUATION 2 — SEMANTIC SIMILARITY

Query                                                     Score
-----------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What are recent trends in computer vision research?      0.4849  █████████


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


How is NLP used in text summarization and classificat    0.6543  █████████████


...

What are the main gaps in bioinformatics research?       0.5761  ███████████


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Latest advances in robotics and deep learning?           0.6488  ████████████


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


How is machine learning applied in cybersecurity?        0.5624  ███████████
What future directions exist in climate modeling with    0.6490  ████████████

 Average Semantic Similarity: 0.5959
 Rating: GOOD (0.50 – 0.69)


##  Cell 10 — ROUGE Score Evaluation

In [17]:
print('=' * 60)
print(' EVALUATION 3 — ROUGE SCORES')
print('=' * 60)

rouge_test_cases = [
    {
        'query': 'What are recent advances in computer vision?',
        'reference': 'Recent advances in computer vision include transformer-based architectures, self-supervised learning methods, and improved real-time object detection models. Challenges remain in generalization and robustness across diverse environments.'
    },
    {
        'query': 'How is NLP applied in research summarization?',
        'reference': 'NLP models are applied to automatically summarize academic papers, extract key findings, and identify research gaps. Transformer models such as BERT and GPT have significantly improved the quality of text summarization systems.'
    },
    {
        'query': 'What are gaps in cybersecurity machine learning research?',
        'reference': 'Key gaps include the lack of large labeled datasets, difficulty detecting zero-day attacks, and limited interpretability of deep learning models in intrusion detection and malware classification tasks.'
    },
    {
        'query': 'What are future directions in robotics AI research?',
        'reference': 'Future robotics research should focus on sim-to-real transfer, safe reinforcement learning, multi-robot coordination, and developing systems that generalize across unstructured environments.'
    },
]

scorer_rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
r1_scores, r2_scores, rL_scores = [], [], []

print(f"\n{'Query':<48} {'R-1':>7} {'R-2':>7} {'R-L':>7}")
print('-' * 72)

for case in rouge_test_cases:
    response, _ = rag_generate(case['query'])
    scores = scorer_rouge.score(case['reference'], response[:600])
    r1 = scores['rouge1'].fmeasure
    r2 = scores['rouge2'].fmeasure
    rL = scores['rougeL'].fmeasure
    r1_scores.append(r1)
    r2_scores.append(r2)
    rL_scores.append(rL)
    print(f"{case['query'][:46]:<48} {r1:>7.4f} {r2:>7.4f} {rL:>7.4f}")

print('\n' + '=' * 60)
print(f' Avg ROUGE-1 : {np.mean(r1_scores):.4f}')
print(f'Avg ROUGE-2 : {np.mean(r2_scores):.4f}')
print(f'Avg ROUGE-L : {np.mean(rL_scores):.4f}')
print('=' * 60)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 EVALUATION 3 — ROUGE SCORES

Query                                                R-1     R-2     R-L
------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What are recent advances in computer vision?      0.1017  0.0000  0.0847


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


How is NLP applied in research summarization?     0.0462  0.0000  0.0462


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What are gaps in cybersecurity machine learnin    0.0480  0.0000  0.0320
What are future directions in robotics AI rese    0.0571  0.0000  0.0571

 Avg ROUGE-1 : 0.0632
Avg ROUGE-2 : 0.0000
Avg ROUGE-L : 0.0550


##  Cell 11 — Final Summary Report

In [18]:
print('\n')
print('   FINAL EVALUATION REPORT — AI Research Assistant v2')
print(f'  Retrieval Accuracy (Semantic) : {retrieval_accuracy:.2f}%')
print(f'  Avg Semantic Similarity       : {avg_similarity:.4f}')
print(f'  Avg ROUGE-1                   : {np.mean(r1_scores):.4f}')
print(f'  Avg ROUGE-2                   : {np.mean(r2_scores):.4f}')
print(f'  Avg ROUGE-L                   : {np.mean(rL_scores):.4f}')

print()
print('  📌 Suggested CV line:')
print(f'  "Achieved {retrieval_accuracy:.0f}% retrieval accuracy and')
print(f'   {avg_similarity:.2f} avg semantic similarity using BAAI/bge-large"')



   FINAL EVALUATION REPORT — AI Research Assistant v2
  Retrieval Accuracy (Semantic) : 100.00%
  Avg Semantic Similarity       : 0.5959
  Avg ROUGE-1                   : 0.0632
  Avg ROUGE-2                   : 0.0000
  Avg ROUGE-L                   : 0.0550

  📌 Suggested CV line:
  "Achieved 100% retrieval accuracy and
   0.60 avg semantic similarity using BAAI/bge-large"


##  Cell 12 — Launch Gradio App

In [19]:
with gr.Blocks(title='AI Research Assistant ') as demo:
    gr.Markdown('# AI Research Assistant — Improved')
    gr.Markdown('Powered by **BAAI/bge-large-en-v1.5** + **Qwen2.5-7B** + **Cross-Encoder Re-ranking**')

    with gr.Tab('📊 Research Query & Analysis'):
        query_box = gr.Textbox(label='Enter your research question', lines=2,
                               placeholder='e.g., Recent trends in reinforcement learning for robotics')
        run_btn = gr.Button('🔍 Analyze Research')
        answer_box = gr.Textbox(label='AI Summary & Insights', lines=8)
        confidence_table = gr.Dataframe(
            headers=['Paper Title', 'Relevance Score'],
            label='Top Retrieved Papers'
        )
        def run_query(q):
            response, retrieved = rag_generate(q)
            table = [[p['title'], round(p['score'], 4)] for p in retrieved]
            return response, table
        run_btn.click(run_query, query_box, [answer_box, confidence_table])

    with gr.Tab('💬 Research Chatbot'):
        chatbot = gr.Chatbot(label='Chat with AI Assistant')
        msg_box = gr.Textbox(label='Ask a research question')
        state = gr.State([])
        def chat_fn(message, history):
            response, _ = rag_generate(message)
            history.append((message, response))
            return history, history
        msg_box.submit(chat_fn, [msg_box, state], [chatbot, state])

    with gr.Tab('📘 Upload PDF Paper'):
        pdf_file = gr.File(label='Upload a research PDF file')
        analyze_btn = gr.Button('🧩 Analyze PDF')
        pdf_result = gr.Textbox(label='AI Summary of PDF', lines=10)
        analyze_btn.click(lambda f: analyze_pdf(f.name), pdf_file, pdf_result)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1fd70969e5ef99ffaf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
